# Teste: render dos vídeos com placa de vídeo (GPU) no Colab — versão 3

Este notebook pega o vídeo **Elefantes** (que hoje é renderizado sem placa de vídeo, a ~1,3 quadro/s) e mede se o Colab consegue fazer **mais rápido**.

**Como usar:**
1. Menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)** → Salvar.
2. Rode as células na ordem (▶ em cada uma, ou *Ambiente de execução → Executar tudo*).
3. A célula **3** mostra o resultado: quantos quadros por segundo cada modo consegue.
4. Se valer a pena, a célula **4** renderiza o vídeo inteiro e a **5** salva o MP4 no seu Google Drive.


In [ ]:
#@title 1. Preparar — VERSÃO 3 (instala o navegador e baixa o projeto) — ~2 min
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "SEM GPU: troque o tipo de ambiente para GPU"
%pip install -q playwright
!python -m playwright install --with-deps chromium 2>&1 | tail -n 3
import importlib; importlib.invalidate_caches()
from playwright.async_api import async_playwright  # se der erro aqui, rode esta célula de novo
print('playwright ok')
!apt-get -qq install -y ffmpeg > /dev/null
!rm -rf proj && git clone -q --depth 1 -b claude/threejs-shorts-visual-i5binq https://github.com/HROSONE/MENTORIA-JARVIS.git proj
%cd /content/proj/videos/svg
!python build_elef.py && python sfx_elef.py && ls -la elefantes.html elef_sfx.wav
import os; print('processadores:', os.cpu_count())


In [ ]:
#@title 2. Motor de render (definições) — VERSÃO 3
import asyncio, time, os, glob, sys, subprocess, importlib
try:
    from playwright.async_api import async_playwright
except ModuleNotFoundError:
    print('instalando o playwright...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'playwright'], check=True)
    subprocess.run([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'], check=True, capture_output=True)
    importlib.invalidate_caches()
    from playwright.async_api import async_playwright
PAGE = 'file:///content/proj/videos/svg/elefantes.html?hideui&still'
MODOS = {
  'CPU (como hoje)': ['--use-angle=swiftshader', '--enable-unsafe-swiftshader'],
  'GPU (EGL)':       ['--ignore-gpu-blocklist', '--enable-gpu', '--enable-gpu-rasterization', '--use-gl=egl'],
  'GPU (Vulkan)':    ['--ignore-gpu-blocklist', '--enable-gpu', '--enable-gpu-rasterization', '--use-angle=vulkan', '--enable-features=Vulkan'],
}
async def _worker(pw, args, frames, outdir, fps):
    b = await launch(pw, args)
    p = await b.new_page(viewport={'width': 1920, 'height': 1080})
    await p.goto(PAGE, wait_until='domcontentloaded', timeout=600000)
    await p.wait_for_function('window.__ready', timeout=600000)
    await p.evaluate('window.__render = true')
    stage = await p.query_selector('#stage')
    for i in frames:
        fp = f'{outdir}/f{i:05d}.jpg'
        if os.path.exists(fp): continue
        await p.evaluate(f'window.__seek({i / fps})')
        await stage.screenshot(path=fp, type='jpeg', quality=92)
    await b.close()
async def render(args, frames, outdir, fps=24, workers=None):
    os.makedirs(outdir, exist_ok=True)
    workers = workers or max(2, os.cpu_count())
    async with async_playwright() as pw:
        await asyncio.gather(*[_worker(pw, args, frames[w::workers], outdir, fps) for w in range(workers)])
async def launch(pw, args):
    # 'chromium' = modo headless novo (navegador completo, pode usar a GPU); se não houver, usa o padrão
    try:
        return await pw.chromium.launch(channel='chromium', args=args + ['--no-sandbox'])
    except Exception:
        return await pw.chromium.launch(args=args + ['--no-sandbox'])
async def gpu_info(args):
    try:
        async with async_playwright() as pw:
            b = await launch(pw, args); p = await b.new_page()
            r = await p.evaluate("() => { const g = document.createElement('canvas').getContext('webgl'); if (!g) return 'sem WebGL'; const e = g.getExtension('WEBGL_debug_renderer_info'); return e ? g.getParameter(e.UNMASKED_RENDERER_WEBGL) : g.getParameter(g.RENDERER); }")
            await b.close(); return 'placa usada: ' + str(r)
    except Exception as e:
        return 'não deu pra ler a placa (' + str(e)[:80] + ')'
print('ok')


In [ ]:
#@title 3. Teste de velocidade (96 quadros de uma cena pesada)
FPS = 24
quadros = list(range(int(50 * FPS), int(50 * FPS) + 96))   # cena da família, aos 0:50
resultado = {}
for nome, args in MODOS.items():
    out = f'/content/bench_{nome.split()[0]}_{len(resultado)}'
    !rm -rf "{out}"
    try:
        print(f'--- {nome}'); print(await gpu_info(args))
        t0 = time.time(); await render(args, quadros, out, FPS); dt = time.time() - t0
        resultado[nome] = len(quadros) / dt
        print(f'{nome}: {resultado[nome]:.2f} quadros/s\n')
    except Exception as e:
        print(nome, 'falhou:', e)
assert resultado, 'nenhum modo funcionou: mande o print de cima'
melhor = max(resultado, key=resultado.get)
total = 436 * FPS
print('MAIS RÁPIDO:', melhor, f'{resultado[melhor]:.2f} q/s')
print(f'Vídeo inteiro ({total} quadros) levaria ~{total / resultado[melhor] / 60:.0f} min aqui no Colab')
print('Referência: no ambiente atual (sem GPU, 4 processadores) leva ~135 min')


In [ ]:
#@title 4. (Opcional) Renderizar o vídeo inteiro com o modo mais rápido
FPS = 24
await render(MODOS[melhor], list(range(436 * FPS + 1)), '/content/frames', FPS)
!ffmpeg -y -loglevel error -framerate {FPS} -i /content/frames/f%05d.jpg -i elef_sfx.wav -c:v libx264 -crf 18 -pix_fmt yuv420p -c:a aac -b:a 192k -shortest /content/elefantes_colab.mp4
!ls -la /content/elefantes_colab.mp4


In [ ]:
#@title 5. (Opcional) Salvar o MP4 no seu Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/videos_colab && cp /content/elefantes_colab.mp4 /content/drive/MyDrive/videos_colab/
print('Salvo em Meu Drive / videos_colab / elefantes_colab.mp4')
